 # 🛡️ 04: Production Replay Audit & Live Order Generation Engine



 **Two-Phase Architecture:**

 1. **Module A (Deterministic Pre-Flight Gate):** Dynamically binds to the active walk-forward

    generation (`run_metadata.json`), verifying that active fold champion neural networks (Chunk 4)

    reconstruct the observation space and deterministically reproduce ground-truth blotter actions.

 2. **Module B (Production Live Execution Blotter):** Warms up the MTM sleeve queue on trailing

    $H$ sessions, runs multi-seed inference at Day $T$ close, computes target dollar allocations,

    applies integer share discretization (`int(target_dollars // current_price)`), and partitions

    manifests into generation-specific subdirectories (`trades/gen{G}/`) for broker dispatch.

In [1]:
import json
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
from pandas.tseries.holiday import USFederalHolidayCalendar
from pandas.tseries.offsets import CustomBusinessDay

from core.paths import LOCAL_DATA_DIR, OUTPUT_DIR, TRADES_DIR
from core.settings import TradingConfig
from data_pipeline.loader import load_processed_data
from rl_discovery.adapter import make_eval_env
from rl_discovery.agent import AbsoluteZeroAgent
from rl_discovery.validator import AgentEvaluator
from run_walk_forward import get_master_trading_calendar


# =====================================================================
# 1. DYNAMIC RUN METADATA BINDING
# =====================================================================
def load_authoritative_metadata(
    preferred_gen: int = 25,
) -> Tuple[Dict[str, Any], Path, Path]:
    """Dynamically locates and loads the authoritative generation metadata and its parent directory."""
    target_dir = OUTPUT_DIR / "model_checkpoints" / f"walk_forward_gen{preferred_gen}"
    meta_path = target_dir / "run_metadata.json"

    if not meta_path.exists():
        candidates = list(OUTPUT_DIR.rglob("run_metadata*.json"))
        if not candidates:
            raise FileNotFoundError(f"❌ No run_metadata.json found in {OUTPUT_DIR}!")
        gen_matches = [c for c in candidates if f"gen{preferred_gen}" in str(c)]
        meta_path = gen_matches[0] if gen_matches else candidates[-1]
        target_dir = meta_path.parent

    with open(meta_path, "r") as f:
        meta = json.load(f)

    return meta, meta_path, target_dir


metadata, metadata_file_path, gen_checkpoint_dir = load_authoritative_metadata(
    preferred_gen=25
)

GEN_ID: int = metadata.get("generation", 25)
SEEDS: List[int] = metadata.get("seeds", [42, 101, 777])
CACHE_FILENAME: str = metadata["data_lineage"]["cache_file"]
OBS_DIM: int = metadata["data_lineage"]["obs_dim"]
ACTION_DIM: int = metadata["data_lineage"]["action_dim"]

# Identify active terminal chunk (Chunk 4 for Gen 25)
chunk_specs_list = metadata.get("chunk_specs", [])
chunk_specs = {cs["chunk_id"]: cs for cs in chunk_specs_list}
ACTIVE_CHUNK_ID: int = max(chunk_specs.keys())
active_chunk_spec = chunk_specs[ACTIVE_CHUNK_ID]

print("=" * 85)
print(
    f"🎯 PRODUCTION TARGET: GENERATION {GEN_ID} | ACTIVE FOLD: CHUNK {ACTIVE_CHUNK_ID}"
)
print(f"   • Metadata Path     : {metadata_file_path}")
print(f"   • Checkpoint Folder : {gen_checkpoint_dir}")
print(
    f"   • Deploy Window     : {active_chunk_spec['deploy_start']} -> {active_chunk_spec.get('deploy_end', 'Present (Terminal Frontier)')}"
)
print(f"   • Feature Cache     : {CACHE_FILENAME}")
print("=" * 85)

NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3
PROJECT_ROOT: C:\Users\ping\Files_win10\python\py311\stocks
GLOBAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data
GLOBAL_PROCESSED_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data\processed
LOCAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\data
OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\output
TRADES_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\trades

🎯 PRODUCTION TARGET: GENERATION 25 | ACTIVE FOLD: CHUNK 4
   • Metadata Path     : C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\output\model_checkpoints\walk_forward_gen25\run_metadata.json
   • Checkpoint Folder : C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\output\model_checkpoints\walk_forward_gen25
   • Deploy Window     : 2026-03-25 -> None
   • Feature Cache     : alpha_cache_41d_2015.parquet


In [2]:
# =====================================================================
# 2. MODULE A: DETERMINISTIC CHECKPOINT REPLAY AUDIT (CHUNK 4 GATE)
# =====================================================================
print("\n" + "=" * 85)
print(f"🛡️ MODULE A: DETERMINISTIC REPLAY AUDIT (CHUNK {ACTIVE_CHUNK_ID} DEPLOYMENT)")
print("=" * 85)

config = TradingConfig()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Ingest Data & Lineage-Bound Features
data = load_processed_data()
df_close = data.df_ohlcv["Adj Close"].unstack(level=0).sort_index()
master_cal = get_master_trading_calendar(data.df_ohlcv, config.calendar_ticker)

feature_cube_path = LOCAL_DATA_DIR / CACHE_FILENAME
if not feature_cube_path.exists():
    raise FileNotFoundError(f"❌ Feature cache {feature_cube_path} does not exist!")
feature_cube = pd.read_parquet(feature_cube_path)

valid_dates = set(feature_cube.index.get_level_values("Date").unique())
trading_calendar = master_cal[master_cal.isin(valid_dates)]

simple_ret_matrix = df_close.pct_change(1, fill_method=None).shift(-1)
simple_ret_matrix["CASH"] = 0.0

# Dynamic Chunk 4 deployment calendar slice
chunk_deploy_start = pd.Timestamp(active_chunk_spec["deploy_start"])
chunk_deploy_end = (
    pd.Timestamp(active_chunk_spec["deploy_end"])
    if active_chunk_spec.get("deploy_end") is not None
    else trading_calendar.max()
)

cal_deploy_active = trading_calendar[
    (trading_calendar >= chunk_deploy_start) & (trading_calendar <= chunk_deploy_end)
]

TIER1_WEIGHT_TOL = 1e-6
TIER2_MAX_WEIGHT_DELTA = 5.0e-3
TIER2_MEAN_WEIGHT_DELTA = 2.5e-4
TIER2_MIN_JACCARD_OVERLAP = 0.85
TIER2_MIN_RETURN_CORR = 0.995

for s in SEEDS:
    # 2. Scoped Checkpoint Resolution (Strictly within Active Generation Directory)
    ckpt_path = gen_checkpoint_dir / f"model_chunk{ACTIVE_CHUNK_ID}_s{s}_champion.pt"
    if not ckpt_path.exists():
        raise FileNotFoundError(f"❌ Checkpoint not found: {ckpt_path}")

    # 3. Scoped Ground-Truth Parquet Blotter Resolution
    blotter_path = OUTPUT_DIR / f"blotter_continuous_s{s}_gen{GEN_ID}.parquet"
    if not blotter_path.exists():
        blotter_path = (
            OUTPUT_DIR
            / "canonical_anchors"
            / f"blotter_continuous_s{s}_gen{GEN_ID}.parquet"
        )
    if not blotter_path.exists():
        raise FileNotFoundError(f"❌ Ground-truth blotter not found: {blotter_path}")

    payload = torch.load(ckpt_path, map_location=device, weights_only=False)
    agent = AbsoluteZeroAgent(obs_dim=OBS_DIM, action_dim=ACTION_DIM).to(device)
    agent.load_state_dict(payload["model_state_dict"])
    agent.eval()

    # 4. Build Evaluation Gym Environment
    gym_eval = make_eval_env(
        feature_cube=feature_cube,
        simple_ret_matrix=simple_ret_matrix,
        calendar=cal_deploy_active,
        macro_df=data.macro_df,
        config=config,
        scaler_state=payload["scaler_state"],
    )

    # 5. Roll Out Policy Replay
    eval_res = AgentEvaluator.evaluate(
        agent, gym_eval, device=device, detailed_log=True
    )
    replayed_blotter = pd.DataFrame(eval_res["blotter"])
    replayed_blotter["date"] = pd.to_datetime(replayed_blotter["date"])

    # 6. Ingest Ground-Truth Parquet Blotter Slice
    s_df = pd.read_parquet(blotter_path)
    s_df["date"] = pd.to_datetime(s_df.get("date", s_df.get("decision_date")))
    s_df = s_df.sort_values("date").reset_index(drop=True)

    stored_slice = (
        s_df[s_df["date"] >= chunk_deploy_start]
        .reset_index(drop=True)
        .iloc[: len(replayed_blotter)]
    )

    # 7. Compute Audit Metrics
    total_sessions = len(replayed_blotter)
    w_replay = replayed_blotter["weight_active"].to_numpy()
    w_stored = stored_slice["weight_active"].to_numpy()
    delta_w_act = float(np.max(np.abs(w_replay - w_stored)))
    mean_w_act = float(np.mean(np.abs(w_replay - w_stored)))

    r_replay = replayed_blotter["net_daily_simple_ret"].to_numpy()
    r_stored = stored_slice["net_daily_simple_ret"].to_numpy()
    delta_net_ret = float(np.max(np.abs(r_replay - r_stored)))

    std_replay, std_stored = np.std(r_replay), np.std(r_stored)
    corr_net_ret = (
        float(np.corrcoef(r_replay, r_stored)[0, 1])
        if (std_replay > 1e-12 and std_stored > 1e-12)
        else 1.0
    )

    jaccard_scores = []
    mismatched_sessions = 0
    mismatch_details = []

    for i in range(total_sessions):
        r_tkrs = set(replayed_blotter["tickers"].iloc[i])
        s_tkrs = set(stored_slice["tickers"].iloc[i])

        union = len(r_tkrs | s_tkrs)
        j_score = len(r_tkrs & s_tkrs) / union if union > 0 else 1.0
        jaccard_scores.append(j_score)

        if r_tkrs != s_tkrs:
            mismatched_sessions += 1
            dt_str = replayed_blotter["date"].iloc[i].strftime("%Y-%m-%d")
            mismatch_details.append((dt_str, r_tkrs - s_tkrs, s_tkrs - r_tkrs))

    mean_jaccard = float(np.mean(jaccard_scores))
    exact_match_pct = (total_sessions - mismatched_sessions) / total_sessions

    print(f"\n🌱 Seed {s:03d} Policy Replay Results ({total_sessions} sessions):")
    print(
        f"  • Weight Active Parity Delta : Max: {delta_w_act:.2e} | Mean: {mean_w_act:.2e}"
    )
    print(
        f"  • Net Return Parity          : Max Δ: {delta_net_ret:.2e} | Correlation: {corr_net_ret:.6f}"
    )
    print(
        f"  • Basket Jaccard Overlap     : {mean_jaccard * 100:.2f}% (Exact Match: {exact_match_pct * 100:.1f}%)"
    )

    # 8. Two-Tier Verification Gate Evaluation
    if delta_w_act < TIER1_WEIGHT_TOL and mismatched_sessions == 0:
        print(
            "  • Gate Status                : ✅ TIER 1 PASSED (Bit-for-Bit Machine Parity)"
        )
    else:
        if mismatch_details:
            print(
                f"  ℹ️ Notice: {len(mismatch_details)} session(s) swapped marginal stock:"
            )
            for dt, in_tkrs, out_tkrs in mismatch_details[:3]:
                print(f"     - {dt}: Swapped {out_tkrs} ➜ {in_tkrs}")

        assert (
            delta_w_act < TIER2_MAX_WEIGHT_DELTA
        ), f"Seed {s}: Max active weight delta {delta_w_act:.2e} >= {TIER2_MAX_WEIGHT_DELTA:.2e}!"
        assert (
            mean_w_act < TIER2_MEAN_WEIGHT_DELTA
        ), f"Seed {s}: Mean active weight delta {mean_w_act:.2e} >= {TIER2_MEAN_WEIGHT_DELTA:.2e}!"
        assert (
            mean_jaccard >= TIER2_MIN_JACCARD_OVERLAP
        ), f"Seed {s}: Basket Jaccard {mean_jaccard*100:.2f}% < {TIER2_MIN_JACCARD_OVERLAP*100:.1f}%!"
        assert (
            corr_net_ret >= TIER2_MIN_RETURN_CORR
        ), f"Seed {s}: Return correlation {corr_net_ret:.6f} < {TIER2_MIN_RETURN_CORR:.4f}!"
        print(
            "  • Gate Status                : ✅ TIER 2 PASSED (Production Restatement Invariants Verified)"
        )

print("\n" + "=" * 85)
print(
    "🏆 FULL-STACK AUDIT PASSED: Active Checkpoint Weights & Scaler States Validated!"
)
print("=" * 85)


🛡️ MODULE A: DETERMINISTIC REPLAY AUDIT (CHUNK 4 DEPLOYMENT)

🌱 Seed 042 Policy Replay Results (126 sessions):
  • Weight Active Parity Delta : Max: 4.77e-08 | Mean: 1.11e-08
  • Net Return Parity          : Max Δ: 5.21e-03 | Correlation: 0.988124
  • Basket Jaccard Overlap     : 81.26% (Exact Match: 0.0%)
  ℹ️ Notice: 126 session(s) swapped marginal stock:
     - 2026-03-25: Swapped {'GNRC', 'DUOL'} ➜ set()
     - 2026-03-26: Swapped {'XLV', 'MET'} ➜ set()
     - 2026-03-27: Swapped {'TER', 'BIDU'} ➜ set()


AssertionError: Seed 42: Basket Jaccard 81.26% < 85.0%!

In [ ]:
# =====================================================================
# 3. MODULE B: LIVE ORDER GENERATION & INTEGER SHARE BLOTTER ENGINE
# =====================================================================
def generate_production_orders(
    portfolio_equity: float = 100_000.0,
    seeds: List[int] = SEEDS,
    chunk_id: int = ACTIVE_CHUNK_ID,
    gen_id: int = GEN_ID,
) -> pd.DataFrame:
    """Computes deterministic trade orders, discretizes dollar allocations into integer shares,
    and exports an execution manifest and thinkorswim Basket Trader CSV to a generation-scoped directory.
    """
    cfg = TradingConfig()
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    num_seeds = len(seeds)
    seed_capital = portfolio_equity / num_seeds
    sleeve_capital = seed_capital / cfg.holding_period

    # Enforce Generation-Scoped Artifact Directory
    gen_trades_dir = TRADES_DIR / f"gen{gen_id}"
    gen_trades_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 85)
    print(f"💼 PRODUCTION TRADING ENGINE: PORTFOLIO EQUITY = ${portfolio_equity:,.2f}")
    print(f"  • Generation                    : Gen {gen_id}")
    print(
        f"  • Active Model Checkpoint       : Chunk {chunk_id} (Active Deployment Fold)"
    )
    print(f"  • Total Number of Model Seeds   : {num_seeds} ({seeds})")
    print(f"  • Capital Allocated per Seed    : ${seed_capital:,.2f}")
    print(
        f"  • Daily Sleeve Allocation (1/H) : ${sleeve_capital:,.2f} per model (H={cfg.holding_period})"
    )
    print(f"  • Output Directory              : {gen_trades_dir}")
    print("=" * 85)

    # Decision Date T (latest available session)
    decision_date = trading_calendar[-1]
    decision_date_str = decision_date.strftime("%Y-%m-%d")

    # Explicit Execution Date T+1 (Next valid NYSE trading session)
    future_cal = master_cal[master_cal > decision_date]
    if len(future_cal) > 0:
        exec_date = future_cal[0]
    else:
        us_bd = CustomBusinessDay(calendar=USFederalHolidayCalendar())
        exec_date = decision_date + us_bd
    exec_date_str = exec_date.strftime("%Y-%m-%d")

    # Oldest active sleeve date to liquidate (entered H sessions ago)
    prior_sessions = master_cal[master_cal <= decision_date]
    liquidate_exec_date = (
        prior_sessions[-cfg.holding_period]
        if len(prior_sessions) >= cfg.holding_period
        else prior_sessions[0]
    )
    liquidate_date_str = liquidate_exec_date.strftime("%Y-%m-%d")

    # Trailing H-session warm-up calendar
    eval_calendar = trading_calendar[-cfg.holding_period :]

    # Terminal live date zero-padding to eliminate lookahead return bias
    ret_matrix = simple_ret_matrix.copy()
    ret_matrix.loc[decision_date] = ret_matrix.loc[decision_date].fillna(0.0)

    seed_orders = {}

    for s in seeds:
        c_path = gen_checkpoint_dir / f"model_chunk{chunk_id}_s{s}_champion.pt"
        c_payload = torch.load(c_path, map_location=dev, weights_only=False)

        ag = AbsoluteZeroAgent(obs_dim=OBS_DIM, action_dim=ACTION_DIM).to(dev)
        ag.load_state_dict(c_payload["model_state_dict"])
        ag.eval()

        gym_env = make_eval_env(
            feature_cube=feature_cube,
            simple_ret_matrix=ret_matrix,
            calendar=eval_calendar,
            macro_df=data.macro_df,
            config=cfg,
            scaler_state=c_payload["scaler_state"],
        )

        res = AgentEvaluator.evaluate(ag, gym_env, device=dev, detailed_log=True)
        last_step = res["blotter"][-1]

        selected_tkrs = list(last_step["tickers"])
        w_active = float(last_step["weight_active"])
        w_bm = float(last_step["weight_benchmark"])

        active_dollars = sleeve_capital * w_active
        bm_dollars = sleeve_capital * w_bm
        per_stock_dollars = (
            active_dollars / len(selected_tkrs) if len(selected_tkrs) > 0 else 0.0
        )

        seed_orders[s] = {
            "tickers": selected_tkrs,
            "w_active": w_active,
            "w_bm": w_bm,
            "stock_dollars": per_stock_dollars,
            "benchmark_dollars": bm_dollars,
        }

        print(f"\n🌱 Seed {s:03d} Model Output for Decision Date {decision_date_str}:")
        print(
            f"  • Asset Allocation : {w_active*100:.1f}% Active | {w_bm*100:.1f}% Benchmark ({cfg.benchmark})"
        )
        print(f"  • Selected Basket  : {selected_tkrs}")
        print(
            f"  • $ / Tkr Alloc    : ${per_stock_dollars:,.2f} each across {len(selected_tkrs)} stocks"
        )
        print(f"  • $ Benchmark Alloc: ${bm_dollars:,.2f}")

    # Net Required Buys Across All Seeds
    net_dollar_orders: Dict[str, float] = {}
    for s, d in seed_orders.items():
        for tkr in d["tickers"]:
            net_dollar_orders[tkr] = (
                net_dollar_orders.get(tkr, 0.0) + d["stock_dollars"]
            )
        net_dollar_orders[cfg.benchmark] = (
            net_dollar_orders.get(cfg.benchmark, 0.0) + d["benchmark_dollars"]
        )

    # Discretize into Integer Shares
    order_rows = []
    latest_prices = df_close.loc[decision_date]

    for tkr, target_dollars in net_dollar_orders.items():
        if target_dollars <= 0.0:
            continue
        current_px = float(latest_prices.get(tkr, np.nan))
        if np.isnan(current_px) or current_px <= 0.0:
            current_px = float(df_close[tkr].dropna().iloc[-1])

        # Blotter Rule: Discrete integer share conversion
        integer_shares = int(np.floor(target_dollars / current_px))
        allocated_dollars = integer_shares * current_px
        residual_cash = target_dollars - allocated_dollars

        order_rows.append(
            {
                "Ticker": tkr,
                "Action": "BUY",
                "Order_Type": "MOC",
                "Shares": integer_shares,
                "Ref_Close_Price": round(current_px, 2),
                "Target_Buy_Dollars": round(target_dollars, 2),
                "Allocated_Dollars": round(allocated_dollars, 2),
                "Residual_Cash": round(residual_cash, 2),
            }
        )

    df_orders = (
        pd.DataFrame(order_rows)
        .sort_values(by="Allocated_Dollars", ascending=False)
        .reset_index(drop=True)
    )

    total_target_dlrs = df_orders["Target_Buy_Dollars"].sum()
    total_allocated_dlrs = df_orders["Allocated_Dollars"].sum()
    total_residual = df_orders["Residual_Cash"].sum()

    print("\n" + "=" * 90)
    print(f"📋 NET MOC DISCRETE BUY ORDERS (FOR EXECUTION AT CLOSE ON {exec_date_str})")
    print("=" * 90)
    print(
        f"{'Ticker':<8} {'Action':<6} {'Shares':>8} {'Ref Price':>11} "
        f"{'Target $':>13} {'Allocated $':>13} {'Residual $':>11}"
    )
    print("-" * 90)
    for _, r in df_orders.iterrows():
        print(
            f"{r['Ticker']:<8} {r['Action']:<6} {int(r['Shares']):>8d} ${r['Ref_Close_Price']:>10.2f} "
            f"${r['Target_Buy_Dollars']:>12,.2f} ${r['Allocated_Dollars']:>12,.2f} ${r['Residual_Cash']:>10.2f}"
        )
    print("-" * 90)
    print(
        f"{'TOTALS':<15} {int(df_orders['Shares'].sum()):>8d} {'':>11} "
        f"${total_target_dlrs:>12,.2f} ${total_allocated_dlrs:>12,.2f} ${total_residual:>10.2f}"
    )
    print("=" * 90)
    print("⚠️ EXECUTION INVARIANTS:")
    print(
        f"  1. Submit netted MOC orders by 3:50 PM on {exec_date_str} for Market-On-Close fill."
    )
    print(
        f"  2. Fully liquidate the active sleeve entered on {liquidate_date_str} (H={cfg.holding_period} sessions ago)."
    )
    print("=" * 90)

    # Persist Execution Manifest, Parquet Blotter, and thinkorswim Basket CSV in gen-specific directory
    timestamp_key = decision_date.strftime("%Y%m%d")

    manifest_path = gen_trades_dir / f"execution_manifest_{timestamp_key}.json"
    parquet_path = gen_trades_dir / f"production_orders_{timestamp_key}.parquet"
    tos_csv_path = gen_trades_dir / f"thinkorswim_basket_{timestamp_key}.csv"
    latest_gen_manifest_path = gen_trades_dir / "latest_execution_manifest.json"
    latest_global_manifest_path = TRADES_DIR / "latest_execution_manifest.json"

    # thinkorswim Basket Trader standard schema: Symbol, Action, Quantity, OrderType, LimitPrice
    df_tos = pd.DataFrame(
        {
            "Symbol": df_orders["Ticker"],
            "Action": df_orders["Action"],
            "Quantity": df_orders["Shares"],
            "OrderType": "MOC",
            "LimitPrice": "",
        }
    )
    df_tos.to_csv(tos_csv_path, index=False)
    df_orders.to_parquet(parquet_path, index=False)

    manifest_payload = {
        "decision_date": decision_date_str,
        "execution_date": exec_date_str,
        "liquidate_sleeve_date": liquidate_date_str,
        "generation": gen_id,
        "chunk_id": chunk_id,
        "portfolio_equity": portfolio_equity,
        "holding_period": cfg.holding_period,
        "benchmark": cfg.benchmark,
        "total_target_dollars": float(total_target_dlrs),
        "total_allocated_dollars": float(total_allocated_dlrs),
        "total_residual_cash": float(total_residual),
        "relative_trades_dir": f"trades/gen{gen_id}",
        "orders_parquet": str(parquet_path.name),
        "tos_basket_csv": str(tos_csv_path.name),
        "orders": df_orders.to_dict(orient="records"),
        "seed_breakdown": {
            str(s): {
                "w_active": d["w_active"],
                "w_bm": d["w_bm"],
                "tickers": d["tickers"],
                "stock_dollars": d["stock_dollars"],
                "benchmark_dollars": d["benchmark_dollars"],
            }
            for s, d in seed_orders.items()
        },
    }

    # Save to generation folder
    with open(manifest_path, "w") as f:
        json.dump(manifest_payload, f, indent=2)
    with open(latest_gen_manifest_path, "w") as f:
        json.dump(manifest_payload, f, indent=2)

    # Maintain root latest pointer for orchestrator/service backward compatibility
    with open(latest_global_manifest_path, "w") as f:
        json.dump(manifest_payload, f, indent=2)

    print(
        f"💾 Saved Execution Manifest    : {manifest_path.relative_to(TRADES_DIR.parent)}"
    )
    print(
        f"💾 Saved Production Orders     : {parquet_path.relative_to(TRADES_DIR.parent)}"
    )
    print(
        f"💾 Saved TOS Basket Trader CSV : {tos_csv_path.relative_to(TRADES_DIR.parent)}"
    )
    print(
        f"💾 Updated Gen Latest Pointer  : {latest_gen_manifest_path.relative_to(TRADES_DIR.parent)}"
    )
    print(
        f"💾 Updated Global Pointer      : {latest_global_manifest_path.relative_to(TRADES_DIR.parent)}"
    )

    return df_orders

In [ ]:
# Run Production Execution Pipeline
if __name__ == "__main__":
    orders_df = generate_production_orders(portfolio_equity=100_000.0)